# 02 · Especialização com LoRA (QLoRA 4 bits)

**Objetivo:** especializar o `Qwen2.5-1.5B-Instruct` para o comportamento que o AgroFácil precisa no pipeline RAG:

1. responder **só com JSON válido** no formato do sistema;
2. **citar o id** do trecho usado;
3. **recusar** ("não consta") quando os trechos recuperados não trazem a resposta;
4. usar linguagem simples, para o produtor.

**Dados de treino:** `base/treino_lora.jsonl`, com perguntas escritas pela equipe a partir de cada trecho da base, mais perguntas fora da base com resposta "não consta". Nenhuma pergunta de treino aparece no conjunto de teste (verificado abaixo). Cada exemplo é montado **com o mesmo prompt e os mesmos 3 trechos recuperados** que o sistema usa em produção, para que o modelo aprenda a ler o contexto e não a decorar respostas.

A comparação LoRA × RAG simples é feita no notebook `04_avaliacao.ipynb`.

In [ ]:
# Configuração do ambiente
# No Colab, o projeto fica no Google Drive para que todos os notebooks (01, 02, 04, 03)
# usem os mesmos arquivos: o limiar salvo no 01 e o adaptador LoRA treinado no 02.
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/palomari/agrofacil-genai.git"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive")
except ImportError:
    pass  # execução local

if not Path("src/agrofacil.py").exists():
    if Path("../src/agrofacil.py").exists():
        os.chdir("..")
    else:
        if not Path("agrofacil-genai").exists():
            subprocess.run(["git", "clone", REPO], check=True)
        os.chdir("agrofacil-genai")

assert Path("src/agrofacil.py").exists(), (
    "Versão antiga do repositório nesta pasta. Apague a pasta agrofacil-genai do Drive e rode de novo.")
sys.path.insert(0, os.getcwd())
print("Raiz do projeto:", os.getcwd())
print("Versão:", subprocess.run(["git", "log", "-1", "--format=%h %cd"], capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q -r requirements.txt

## 1. Montagem do conjunto de treino

In [ ]:
import json, random
import pandas as pd
from datasets import Dataset
from src import agrofacil as af

trechos = af.carregar_base()
por_id = {t["id"]: t for t in trechos}
treino = af.carregar_jsonl(af.CAMINHO_TREINO)
testes = af.carregar_testes()

sobrepostas = {af.normalizar(t["pergunta"]) for t in treino} & {af.normalizar(c["pergunta"]) for c in testes}
assert not sobrepostas, f"Pergunta de teste no treino: {sobrepostas}"

cfg = af.carregar_config()
buscador = af.Buscador(trechos, cfg["encoder"])
rng = random.Random(42)

exemplos, forcados = [], 0
for ex in treino:
    rec = buscador.buscar(ex["pergunta"], k=cfg.get("k", 3))
    if ex["fonte_id"] and ex["fonte_id"] not in [t["id"] for t in rec]:
        rec = [por_id[ex["fonte_id"]]] + rec[:-1]   # garante o trecho correto no contexto
        forcados += 1
    rng.shuffle(rec)                                 # evita o atalho "citar sempre o 1º trecho"
    alvo = {"resposta": ex["resposta"], "fonte_id": ex["fonte_id"],
            "status": "ok" if ex["fonte_id"] else "nao_consta"}
    exemplos.append({
        "prompt": af.mensagens_rag(ex["pergunta"], rec),
        "completion": [{"role": "assistant", "content": json.dumps(alvo, ensure_ascii=False)}],
    })

ds = Dataset.from_list(exemplos)
print(f"Exemplos: {len(ds)} ({sum(e['fonte_id'] is not None for e in treino)} respostas, "
      f"{sum(e['fonte_id'] is None for e in treino)} recusas). Trecho correto inserido à força em {forcados}.")
print(json.dumps(exemplos[0], ensure_ascii=False, indent=1)[:1500])

## 2. Treinamento QLoRA

- Modelo base em 4 bits (NF4, `bitsandbytes`), para caber em uma GPU T4.
- LoRA com `r=8`, `alpha=16` nas projeções de atenção e MLP.
- Formato *prompt-completion*: a perda é calculada **apenas na resposta** (o JSON), e não no prompt.
- Semente fixa (42), para ser reprodutível.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(af.MODELO_BASE)
model = AutoModelForCausalLM.from_pretrained(
    af.MODELO_BASE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
    device_map="auto",
)
# o SFTTrainer prepara o modelo 4 bits para treino (prepare_model_for_kbit_training)

peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
args = SFTConfig(
    output_dir="resultados_lora",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=1,
    max_length=1024,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds,
                     peft_config=peft_config, processing_class=tok)
trainer.model.print_trainable_parameters()
trainer.train()

In [ ]:
# Curva de perda e salvamento do adaptador
import matplotlib.pyplot as plt

log = pd.DataFrame(trainer.state.log_history)
log.to_csv("avaliacao/treino_lora_log.csv", index=False)
perdas = log.dropna(subset=["loss"])
plt.figure(figsize=(7, 3))
plt.plot(perdas["step"], perdas["loss"], marker="o", color="#2e8b57")
plt.xlabel("passo"); plt.ylabel("perda (só na resposta)"); plt.title("Treino LoRA")
plt.tight_layout(); plt.savefig("avaliacao/treino_lora_loss.png", dpi=120); plt.show()

af.CAMINHO_ADAPTER.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(af.CAMINHO_ADAPTER)
tok.save_pretrained(af.CAMINHO_ADAPTER)
tamanho = sum(p.stat().st_size for p in af.CAMINHO_ADAPTER.rglob("*") if p.is_file()) / 1e6
print(f"Adaptador salvo em {af.CAMINHO_ADAPTER} ({tamanho:.1f} MB)")
print(f"Perda inicial {perdas['loss'].iloc[0]:.3f} -> final {perdas['loss'].iloc[-1]:.3f}")

## 3. Teste rápido do modelo especializado (perguntas novas, fora do treino e do teste)

In [ ]:
modelo_lora = trainer.model.eval()
modelo_lora.config.use_cache = True
for p in ["Na noite de geada, posso deixar o pivô desligado até o amanhecer?",
          "Qual o preço do milho hoje na bolsa?"]:
    r = af.responder(p, modelo_lora, tok, buscador=buscador, modo="rag", limiar=None)
    print(p)
    print(json.dumps(af.saida_publica(r), ensure_ascii=False, indent=1))
    print("bruto:", r["bruto"]); print("-" * 80)

## 4. Publicar o adaptador

O adaptador fica em `modelos/agrofacil_adapter/` (algumas dezenas de MB). Para que o avaliador reproduza o notebook 04 e a interface sem treinar de novo, faça o commit dessa pasta no repositório (cada arquivo precisa ter menos de 100 MB) ou publique no Hugging Face Hub e ajuste `CAMINHO_ADAPTER`.

```python
# Opcional: baixar o adaptador do Colab
!zip -r agrofacil_adapter.zip modelos/agrofacil_adapter
from google.colab import files; files.download("agrofacil_adapter.zip")
```